# N-gram Language Models and Smoothing: build it, count it, smooth it

This notebook is the runnable companion to the concept page **N-gram Language Models and Smoothing**.
Every function and corpus is **imported from `ngram_lm.py`** (the single source of truth), so the
numbers here, the numbers on the page, and the figures all come from the *same* code and cannot drift.

We will, one cell at a time:
1. count bigrams and compute an MLE probability by hand;
2. watch one unseen bigram zero a whole sentence (the catastrophe);
3. rescue it with Laplace add-one;
4. estimate the Good-Turing missing mass from the singletons;
5. compute Kneser-Ney **continuation** probabilities (versatility, not frequency);
6. train an interpolated-Kneser-Ney trigram, measure **perplexity**, and **generate** text;
7. watch perplexity turn with `n` (the bias-variance turn) and compare Laplace vs Kneser-Ney.

Each cell **asserts its qualitative point first**, then prints the evidence. Pure-Python/NumPy, CPU,
fully deterministic given the seed.


## Setup — import the single source of truth

`ngram_lm.py` lives next to this notebook. It holds the corpora, the counting machinery, and every
smoother. We import from it rather than redefine anything, so this notebook can never disagree with
the page or the figures.

In [1]:
import math
import random

import ngram_lm as ng

print("device: cpu (pure-Python/numpy)")
print("numpy:", __import__("numpy").__version__)
try:
    import torch
    print("torch:", torch.__version__, "(imported for the version line only; n-grams are count-based)")
except ImportError:
    print("torch: not installed (not needed)")
print("seed:", ng.SEED)

device: cpu (pure-Python/numpy)
numpy: 2.4.6


torch: 2.12.0 (imported for the version line only; n-grams are count-based)
seed: 0


## Step 1 — count bigrams and compute an MLE probability by hand

We tokenize the classic SLP3 toy corpus, count bigrams and their contexts, and compute
$P(\text{am}\mid\text{I}) = c(\text{I},\text{am})/c(\text{I})$. We **assert** it equals $2/3$
before printing — the page's Worked Example 1.

In [2]:
toy = ng.tokenize_toy(ng.TOY_CORPUS)
bi = ng.count_ngrams(toy, 2)
ctx_bi = ng.context_counts(bi)

p_am_given_i = ng.mle_conditional("am", ("I",), bi, ctx_bi)
assert abs(p_am_given_i - 2 / 3) < 1e-12, "P(am|I) must be the count ratio 2/3"
print(f"c(I, am) = {bi[('I', 'am')]},  c(I) = {ctx_bi[('I',)]}")
print(f"P(am | I) = c(I,am)/c(I) = {p_am_given_i:.3f}   (= 2/3)")

c(I, am) = 2,  c(I) = 3
P(am | I) = c(I,am)/c(I) = 0.667   (= 2/3)


## Step 2 — the whole sentence, and the zero-probability catastrophe

A sentence's probability is a **product** of bigram probabilities. First we score the fully-seen
`<s> I am Sam </s>` (a healthy 0.111). Then we score a sentence containing the **unseen** pair
`(Sam, do)` and assert the product collapses to exactly **0** — one missing bigram zeros everything.

In [3]:
def bigram_sentence_prob(pairs):
    p = 1.0
    for a, b in pairs:
        p *= ng.mle_conditional(b, (a,), bi, ctx_bi)
    return p

seen = [("<s>", "I"), ("I", "am"), ("am", "Sam"), ("Sam", "</s>")]
unseen = [("<s>", "Sam"), ("Sam", "do"), ("do", "not"), ("not", "</s>")]

p_seen = bigram_sentence_prob(seen)
p_unseen = bigram_sentence_prob(unseen)

assert abs(p_seen - 4 / 36) < 1e-12, "fully-seen sentence should be 4/36 = 0.111"
assert p_unseen == 0.0, "one unseen bigram must zero the whole product"
print(f"P(<s> I am Sam </s>)   = {p_seen:.3f}   (all bigrams seen)")
print(f"P(<s> Sam do not </s>) = {p_unseen:.0f}       (contains unseen (Sam, do) -> ZERO)")
print(f"  culprit: P(do | Sam) = {ng.mle_conditional('do', ('Sam',), bi, ctx_bi):.0f}")

P(<s> I am Sam </s>)   = 0.111   (all bigrams seen)
P(<s> Sam do not </s>) = 0       (contains unseen (Sam, do) -> ZERO)
  culprit: P(do | Sam) = 0


## Step 3 — Laplace add-one rescues the zero

Add-one smoothing pretends we saw every possible next word one extra time:
$P_{\text{Laplace}}(w\mid c) = (c(c,w)+1)/(c(c)+V)$ with $V=12$ on the toy corpus. We assert the
previously-impossible `(Sam, do)` is now exactly $1/14$, and that a seen bigram like `(I, am)` gets
*discounted* (0.667 → 0.2) — the price of moving mass to the unseen.

In [4]:
V = ng.laplace_vocab_size(toy)
assert V == 12, "SLP3 toy vocabulary is 10 content words + <s> + </s> = 12"

p_do_laplace = ng.add_k_conditional("do", ("Sam",), bi, ctx_bi, V, k=1.0)
p_am_laplace = ng.add_k_conditional("am", ("I",), bi, ctx_bi, V, k=1.0)

assert abs(p_do_laplace - 1 / 14) < 1e-12, "Laplace P(do|Sam) = (0+1)/(2+12) = 1/14"
assert p_am_laplace < p_am_given_i, "Laplace must DISCOUNT the seen (I, am) bigram"
print(f"P(do | Sam)  Laplace = (0+1)/(2+12) = {p_do_laplace:.4f}   (was 0 -> rescued)")
print(f"P(am | I)    Laplace = (2+1)/(3+12) = {p_am_laplace:.3f}    (was 0.667 -> discounted)")

P(do | Sam)  Laplace = (0+1)/(2+12) = 0.0714   (was 0 -> rescued)
P(am | I)    Laplace = (2+1)/(3+12) = 0.200    (was 0.667 -> discounted)


In [5]:
# Worked example 6 — Laplace perplexity of the held-out toy sentence "<s> I am Sam </s>"
# (4 bigrams, V=12): PP = 2^(-1/N * sum_i log2 P(w_i | w_{i-1})). This is the "Verified: 5.916" number on the page.
toy_pp_bigrams = [(ng.BOS, "I"), ("I", "am"), ("am", "Sam"), ("Sam", ng.EOS)]
toy_pp_log2 = sum(math.log2(ng.add_k_conditional(w, (p,), bi, ctx_bi, V, k=1.0)) for p, w in toy_pp_bigrams)
toy_pp = 2.0 ** (-toy_pp_log2 / len(toy_pp_bigrams))
assert abs(toy_pp - 5.916) < 1e-2, "toy-sentence Laplace perplexity must be ~5.916"
print(f"Laplace PP('<s> I am Sam </s>', V={V}) = {toy_pp:.3f}   (~6: as uncertain as a uniform choice among ~6 words)")

Laplace PP('<s> I am Sam </s>', V=12) = 5.916   (~6: as uncertain as a uniform choice among ~6 words)


## Step 4 — Good-Turing: the singletons estimate the unseen mass

Good-Turing's headline result is that the probability mass to reserve for **unseen** events is
$N_1/N$ — the fraction of n-grams seen exactly once. We measure it on the larger animal corpus
(where the frequency-of-frequencies is meaningful) and assert it lands near 0.18. We also compute the
reestimated singleton count $c^*(1) = 2\,N_2/N_1$: on a *typical* corpus this discounts a singleton
below 1, but on this tiny, repetitive corpus $N_2$ is unusually close to $N_1$, so $c^*(1)\approx1.71$
— a useful reminder that Good-Turing's reestimates are **noisy on small data**, which is exactly why
practical implementations smooth the $N_r$ first.

In [6]:
animal_all = ng.sentences_from_text(ng.ANIMAL_TEXT)
animal_bi = ng.count_ngrams(animal_all, 2)
nr = ng.frequency_of_frequencies(animal_bi)

missing = ng.good_turing_missing_mass(animal_bi)
c_star_1 = ng.good_turing_reestimated_count(1, nr)

assert nr[1] >= nr[2] >= nr[3], "singletons should dominate the frequency-of-frequencies"
assert 0.10 < missing < 0.30, "N1/N reserves a meaningful slice of mass for the unseen"
print(f"N1={nr[1]}  N2={nr[2]}  N3={nr[3]}   total bigram tokens N={sum(animal_bi.values())}")
print(f"Good-Turing missing mass  N1/N = {nr[1]}/{sum(animal_bi.values())} = {missing:.3f}")
print(f"Good-Turing c*(1) = 2 * N2/N1 = 2*{nr[2]}/{nr[1]} = {c_star_1:.2f}")
print("  (noisy on tiny data: N2 ~ N1 here, so c*(1) > 1 -- why practical GT smooths the N_r first)")

N1=14  N2=12  N3=5   total bigram tokens N=78
Good-Turing missing mass  N1/N = 14/78 = 0.179
Good-Turing c*(1) = 2 * N2/N1 = 2*12/14 = 1.71
  (noisy on tiny data: N2 ~ N1 here, so c*(1) > 1 -- why practical GT smooths the N_r first)


## Step 5 — Kneser-Ney continuation probability: versatility, not frequency

The crux of Kneser-Ney: a word's lower-order score is *how many distinct contexts it completes*,
not how often it occurs. On the toy corpus (15 bigram types) the sentence-ender `</s>` (3 distinct
preceders) should out-score `am` (1 preceder). We assert exactly that — the "Francisco" effect in
miniature.

In [7]:
preceders, n_bi_types = ng.continuation_counts_bigram(toy)
assert n_bi_types == 15, "bigram-order padding gives 15 distinct bigram types on the toy corpus"

p_cont = {w: preceders.get(w, 0) / n_bi_types for w in ("am", "Sam", "</s>")}
assert p_cont["</s>"] > p_cont["am"], "versatile </s> (3 preceders) beats narrow 'am' (1 preceder)"
for w in ("am", "Sam", "</s>"):
    print(f"P_cont({w:<4}) = {preceders.get(w, 0)}/{n_bi_types} = {p_cont[w]:.3f}   "
          f"({preceders.get(w, 0)} distinct preceders)")

P_cont(am  ) = 1/15 = 0.067   (1 distinct preceders)
P_cont(Sam ) = 2/15 = 0.133   (2 distinct preceders)
P_cont(</s>) = 3/15 = 0.200   (3 distinct preceders)


## Step 6 — train a Kneser-Ney trigram, measure perplexity, generate text

Now the full interpolated-Kneser-Ney trigram from `ngram_lm.py`, trained on a seeded train/test
split of the animal corpus. We assert the held-out perplexity is finite and far below the vocabulary
size (the model genuinely learned), then sample two sentences. Note the output is **locally fluent,
globally aimless** — the Markov ceiling you can *see*.

In [8]:
train, test = ng.animal_train_test()
kn = ng.KneserNeyTrigram(train)
pp = kn.perplexity(test)

assert math.isfinite(pp), "smoothing must keep perplexity finite (no zero probabilities)"
assert pp < kn.vocab_size, "a useful model beats uniform guessing over the vocabulary"
print(f"train/test = {len(train)}/{len(test)} sentences, vocab = {kn.vocab_size}")
print(f"KN trigram held-out perplexity = {pp:.2f}   (< vocab {kn.vocab_size} = better than random)")

rng = random.Random(ng.SEED)
print("generated:", kn.generate(rng=rng))
print("generated:", kn.generate(rng=rng))

train/test = 9/3 sentences, vocab = 16
KN trigram held-out perplexity = 5.78   (< vocab 16 = better than random)
generated: a on the dog a
generated: the saw the


## Step 7 — perplexity vs n (the bias-variance turn) and Laplace vs Kneser-Ney

Two measured comparisons, both straight from the canonical models:

- **vs n:** unigram → bigram slashes perplexity (context helps); trigram is the minimum; 4-gram ticks
  back *up* as higher orders starve for counts. We assert the unigram→bigram drop and the eventual rise.
- **vs smoother:** at full data, Kneser-Ney's perplexity is **below** Laplace's — we assert KN wins.

In [9]:
orders = [1, 2, 3, 4]
pps = [ng.KneserNeyOrderN(train, n=n).perplexity(test) for n in orders]
assert pps[0] > pps[1], "adding context (n=1 -> n=2) must lower perplexity sharply"
assert pps[3] >= pps[2], "on a small corpus the highest order starves -> the bias-variance turn"
for n, p in zip(orders, pps):
    print(f"  KN perplexity (n={n}) = {p:.2f}")

laplace_pp = ng.AddKBigram(train, k=1.0).perplexity(test)
kn_pp = ng.KneserNeyOrderN(train, n=2).perplexity(test)
assert kn_pp < laplace_pp, "Kneser-Ney must beat Laplace at full data"
print()
print(f"  Laplace bigram perplexity = {laplace_pp:.2f}")
print(f"  Kneser-Ney bigram perplexity = {kn_pp:.2f}   (KN wins: {kn_pp:.2f} < {laplace_pp:.2f})")

  KN perplexity (n=1) = 12.92
  KN perplexity (n=2) = 5.87
  KN perplexity (n=3) = 5.74
  KN perplexity (n=4) = 5.99

  Laplace bigram perplexity = 8.22
  Kneser-Ney bigram perplexity = 5.87   (KN wins: 5.87 < 8.22)


## Recap

- A language model factorizes $P(W)$ by the **chain rule**; the **Markov assumption** truncates the
  history to the last $n{-}1$ words (the **n-gram**).
- **MLE** is the count ratio $c(\text{ctx},w)/c(\text{ctx})$ — and assigns **0** to any unseen n-gram,
  zeroing whole sentences (Step 2). **Smoothing** is the cure.
- **Laplace** (Step 3) rescues zeros but over-flattens; **Good-Turing** (Step 4) uses the singletons
  ($N_1/N$) to size the unseen mass; **Kneser-Ney** (Step 5) backs off to **versatility, not frequency**
  and is the classical gold standard (Step 6).
- We score with **perplexity** $= 2^{\text{cross-entropy}}$ — the average branching factor, lower is
  better — and watched it **turn** with `n` (Step 7): context helps until the counts go sparse.

Every number above was produced by `ngram_lm.py`; the figures on the page are produced by
`make_figures_04.py`, which imports the **same** functions — so the page, the notebook, and the
figures are one consistent artifact. The same chain-rule / cross-entropy / perplexity machinery
carries straight into neural and transformer LMs; n-grams are the base class they all inherit from.
